In [2]:
data = [1,2,3,4,5,6,7,8,9,10]

In [9]:
train, test = data[0], data[1:]
train, test

(1, [2, 3, 4, 5, 6, 7, 8, 9, 10])

In [10]:
last = train 
forecasts = []
for i in test:
    forecasts.append(last)
    last = i

In [11]:
forecasts

[1, 2, 3, 4, 5, 6, 7, 8, 9]

In [12]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import logging
from sklearn.metrics import (
    mean_absolute_error, 
    mean_squared_error, 
    root_mean_squared_error,
    mean_absolute_percentage_error)

# --- Assumed utils imports ---
# Make sure you have an empty __init__.py file in the 'utils' folder
try:
    from utils.download_data import download_data
    from utils.model_data_prep import prepare_data_for_modeling
except ImportError as e:
    print("="*50)
    print("ERROR: Could not import functions from 'utils' subfolder.")
    print("Please ensure:")
    print("1. You have a subfolder named 'utils'.")
    print("2. It contains an empty file named '__init__.py'.")
    print("3. It contains 'data_download.py' with 'download_data'.")
    print("4. It contains 'model_data_preparation.py' with 'prepare_data_for_modeling'.")
    print(f"Original error: {e}")
    print("="*50)
    exit() # Stop execution if imports fail

# exit()

# --- Configuration ---
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

# --- Parameters ---
TICKER = "MSFT"          # Stock ticker
TIMEFREQ = "1d"          # Data time frequency (e.g., "1d", "1h")
TARGET_COLUMN = 'Close'  # Column to forecast
TEST_SIZE = 0.2          # Proportion of data for the test set (e.g., 0.2 for 80/20 split)
TRAIN_LAST_N = 0.8       # Keep last 80% of the initial train data (use int for absolute count)
PLOT_RESULTS = True      # Whether to display the plot

# Define the base directory (consistent with the utils functions)
BASE_DATA_DIR = Path("data")

# --- Main Execution ---
if __name__ == "__main__":
    logging.info(f"Starting Naive Forecast process for {TICKER} ({TIMEFREQ})")

    # 1. (Optional) Download data if it doesn't exist
    # You can comment this out if you are sure the data is already downloaded
    try:
        logging.info("Checking/Downloading data...")
        # Assuming download_data uses the same BASE_DATA_DIR
        download_data(ticker_input=TICKER, timefreq=TIMEFREQ, base_dir=BASE_DATA_DIR)
    except Exception as e:
        logging.error(f"Error during data download step: {e}")
        # Decide if you want to stop or continue if download fails
        # exit()
    
    # 2. Prepare Data (Load, Split, Truncate)
    logging.info("Preparing data for modeling...")
    train_data = None
    test_data = None
    try:
        prepared_data = prepare_data_for_modeling(
            ticker=TICKER,
            timefreq=TIMEFREQ,
            train_last_n=TRAIN_LAST_N,
            target_column=TARGET_COLUMN,
            test_size=TEST_SIZE,
            base_dir=BASE_DATA_DIR
        )
        if prepared_data:
            train_data, test_data = prepared_data
            logging.info(f"Data prepared: Train size={len(train_data)}, Test size={len(test_data)}")
        else:
            logging.error("Data preparation failed or returned None.")
            exit() # Stop if data preparation fails

    except FileNotFoundError:
        logging.error(f"Required data file not found for {TICKER} ({TIMEFREQ}). Please run download first.")
        exit()
    except ValueError as e:
        logging.error(f"Invalid parameters or data issue during preparation: {e}")
        exit()
    except Exception as e:
        logging.error(f"An unexpected error occurred during data preparation: {e}")
        exit()

    # 3. Perform Naive Forecast
    logging.info("Performing Naive Forecast...")
    if train_data is None or train_data.empty:
        logging.error("Training data is empty, cannot perform Naive forecast.")
        exit()


2025-04-08 12:02:19,396 - INFO - Starting Naive Forecast process for MSFT (1d)
2025-04-08 12:02:19,397 - INFO - Checking/Downloading data...
2025-04-08 12:02:19,397 - INFO - Data for MSFT (1d) already exists at data/MSFT/MSFT_1d.csv. Skipping download.
2025-04-08 12:02:19,398 - INFO - All requested ticker data already exists locally.
2025-04-08 12:02:19,398 - INFO - Preparing data for modeling...
2025-04-08 12:02:19,429 - INFO - Successfully loaded data from data/MSFT/MSFT_1d.csv
2025-04-08 12:02:19,435 - INFO - Initial split: Train size=7875, Test size=1969
2025-04-08 12:02:19,437 - INFO - train_last_n is a float (0.8). Keeping last 6300 points (80.00%) of initial train set.
2025-04-08 12:02:19,438 - INFO - Final train set size after selecting last 6300 points: 6300
2025-04-08 12:02:19,439 - INFO - Data prepared: Train size=6300, Test size=1969
2025-04-08 12:02:19,439 - INFO - Performing Naive Forecast...


In [17]:
train_data.iloc[-1]

66.24838256835938

In [18]:
test_data.head()

Date
2017-06-08    65.845718
2017-06-09    64.354019
2017-06-12    63.859829
2017-06-13    64.656006
2017-06-14    64.308235
Name: Close, dtype: float64

In [27]:
test_data[:-1].values

array([ 65.84571838,  64.35401917,  63.85982895, ..., 382.14001465,
       373.10998535, 359.83999634])

In [29]:
train_data.iloc[-1]

66.24838256835938

In [ ]:
forecasts = [train_data.iloc[-1]]
forecasts.extend(test_data[:-1].values)

In [34]:
forecasts

[66.24838256835938,
 65.84571838378906,
 64.35401916503906,
 63.85982894897461,
 64.656005859375,
 64.30823516845703,
 63.96963882446289,
 64.0611572265625,
 64.85733032226562,
 63.97880935668945,
 64.30823516845703,
 64.29911804199219,
 65.16852569580078,
 64.54620361328125,
 63.33818435668945,
 63.878143310546875,
 62.679290771484375,
 63.08193588256836,
 62.386417388916016,
 63.219215393066406,
 62.75247573852539,
 63.56697463989258,
 64.04286193847656,
 64.0519790649414,
 65.11358642578125,
 65.68099975585938,
 66.60531616210938,
 67.12691497802734,
 67.08119201660156,
 67.59370422363281,
 67.92313385009766,
 67.52960205078125,
 67.3557357788086,
 67.89566802978516,
 67.76753234863281,
 66.95308685302734,
 66.84326171875,
 66.53204345703125,
 66.42230224609375,
 66.1294174194336,
 66.02873992919922,
 66.51380920410156,
 66.25755310058594,
 66.61444854736328,
 66.32160186767578,
 65.35153198242188,
 66.34905242919922,
 67.3465576171875,
 67.36497497558594,
 67.7605972290039,
 66.610